# Agente Analista de Datos con Tool Use

> Construye un agente que responde preguntas en lenguaje natural sobre datasets
> usando el mecanismo de tool_use de Anthropic para ejecutar análisis reales.

## Objetivos
- Definir herramientas de análisis de datos (estadísticas, filtros, gráficas)
- Implementar el bucle agéntico completo con tool_use de la API de Anthropic
- Responder preguntas en lenguaje natural sobre un dataset de ventas
- Visualizar resultados automáticamente

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic pandas matplotlib numpy --quiet

## 2. Dataset de ventas y configuración

In [ ]:
import anthropic
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

client = anthropic.Anthropic()
MODELO = "claude-haiku-4-5-20251001"

# Generar dataset sintético de ventas
np.random.seed(42)
meses = ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
         "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
productos = ["Laptop", "Monitor", "Teclado", "Ratón", "Auriculares"]
regiones = ["Norte", "Sur", "Este", "Oeste"]

filas = []
for mes in meses:
    for producto in productos:
        for region in regiones:
            ventas = int(np.random.normal(100, 30))
            precio = {"Laptop": 999, "Monitor": 349, "Teclado": 79,
                      "Ratón": 49, "Auriculares": 199}[producto]
            filas.append({
                "mes": mes, "producto": producto, "region": region,
                "unidades": max(0, ventas),
                "ingresos": max(0, ventas) * precio
            })

df = pd.DataFrame(filas)
print(f"Dataset creado: {len(df)} registros")
print(df.head())

## 3. Definición de herramientas para el agente

In [ ]:
HERRAMIENTAS = [
    {
        "name": "calcular_estadisticas",
        "description": "Calcula estadísticas agregadas del dataset de ventas (total, media, máx, mín) por columna y grupo opcional.",
        "input_schema": {
            "type": "object",
            "properties": {
                "columna": {"type": "string", "enum": ["unidades", "ingresos"], "description": "Columna a analizar"},
                "agrupar_por": {"type": "string", "enum": ["mes", "producto", "region", "ninguno"], "description": "Cómo agrupar los datos"}
            },
            "required": ["columna", "agrupar_por"]
        }
    },
    {
        "name": "filtrar_datos",
        "description": "Filtra el dataset por un valor específico de mes, producto o región.",
        "input_schema": {
            "type": "object",
            "properties": {
                "campo": {"type": "string", "enum": ["mes", "producto", "region"]},
                "valor": {"type": "string", "description": "Valor a filtrar"}
            },
            "required": ["campo", "valor"]
        }
    },
    {
        "name": "top_n",
        "description": "Devuelve los N productos o regiones con más ventas o ingresos.",
        "input_schema": {
            "type": "object",
            "properties": {
                "metrica": {"type": "string", "enum": ["unidades", "ingresos"]},
                "agrupar_por": {"type": "string", "enum": ["producto", "region", "mes"]},
                "n": {"type": "integer", "default": 3}
            },
            "required": ["metrica", "agrupar_por"]
        }
    }
]

def ejecutar_herramienta(nombre: str, argumentos: dict) -> str:
    """Ejecuta la herramienta solicitada por el agente y devuelve el resultado como string."""
    if nombre == "calcular_estadisticas":
        col = argumentos["columna"]
        grupo = argumentos["agrupar_por"]
        if grupo == "ninguno":
            resultado = df[col].agg(["sum", "mean", "max", "min"]).round(2).to_dict()
        else:
            resultado = df.groupby(grupo)[col].agg(["sum", "mean", "max", "min"]).round(2).to_dict()
        return json.dumps(resultado, ensure_ascii=False)

    elif nombre == "filtrar_datos":
        campo = argumentos["campo"]
        valor = argumentos["valor"]
        filtrado = df[df[campo] == valor]
        resumen = filtrado.groupby("producto")[["unidades", "ingresos"]].sum().round(2)
        return resumen.to_json(force_ascii=False)

    elif nombre == "top_n":
        metrica = argumentos["metrica"]
        grupo = argumentos["agrupar_por"]
        n = argumentos.get("n", 3)
        top = df.groupby(grupo)[metrica].sum().nlargest(n).round(2)
        return top.to_json(force_ascii=False)

    return f"Herramienta {nombre} no reconocida."

print("Herramientas del agente definidas.")

## 4. Bucle agéntico completo

In [ ]:
def agente_analista(pregunta: str) -> str:
    """Agente que responde preguntas sobre el dataset usando herramientas de análisis."""
    print(f"\nPregunta: {pregunta}")
    print("-" * 50)

    mensajes = [{"role": "user", "content": pregunta}]
    sistema = (
        "Eres un analista de datos experto. Tienes acceso a un dataset de ventas con columnas: "
        "mes, producto, region, unidades, ingresos. Usa las herramientas para responder con datos reales. "
        "Productos: Laptop, Monitor, Teclado, Ratón, Auriculares. "
        "Regiones: Norte, Sur, Este, Oeste. Meses: Ene-Dic."
    )

    # Bucle agéntico: el agente llama herramientas hasta tener suficiente información
    for iteracion in range(5):  # máximo 5 llamadas a herramientas
        respuesta = client.messages.create(
            model=MODELO,
            max_tokens=1024,
            system=sistema,
            tools=HERRAMIENTAS,
            messages=mensajes
        )

        if respuesta.stop_reason == "end_turn":
            # El agente ha terminado — extraer respuesta final
            texto_final = next((b.text for b in respuesta.content if hasattr(b, "text")), "")
            print(f"Respuesta: {texto_final}")
            return texto_final

        if respuesta.stop_reason == "tool_use":
            # Ejecutar todas las herramientas solicitadas
            mensajes.append({"role": "assistant", "content": respuesta.content})
            resultados_herramientas = []

            for bloque in respuesta.content:
                if bloque.type == "tool_use":
                    print(f"  → Usando herramienta: {bloque.name}({bloque.input})")
                    resultado = ejecutar_herramienta(bloque.name, bloque.input)
                    resultados_herramientas.append({
                        "type": "tool_result",
                        "tool_use_id": bloque.id,
                        "content": resultado
                    })

            mensajes.append({"role": "user", "content": resultados_herramientas})

    return "El agente no pudo completar el análisis."

# Prueba con diferentes preguntas
agente_analista("¿Cuál es el producto que más ingresos ha generado en total?")

## 5. Más preguntas y visualización

In [ ]:
agente_analista("¿Qué región tiene más unidades vendidas? ¿Y cuál es la media mensual?")

In [ ]:
# Gráfica de ingresos por producto
ingresos_producto = df.groupby("producto")["ingresos"].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Barras de ingresos por producto
colores = ["#3498db", "#2ecc71", "#e74c3c", "#f39c12", "#9b59b6"]
axes[0].bar(ingresos_producto.index, ingresos_producto.values / 1000, color=colores, edgecolor="black", alpha=0.8)
axes[0].set_title("Ingresos Totales por Producto (miles €)", fontweight="bold")
axes[0].set_xlabel("Producto")
axes[0].set_ylabel("Ingresos (k€)")
axes[0].tick_params(axis="x", rotation=20)

# Tendencia mensual de unidades
orden_meses = ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
               "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
unidades_mes = df.groupby("mes")["unidades"].sum().reindex(orden_meses)
axes[1].plot(orden_meses, unidades_mes.values, marker="o", color="#3498db", linewidth=2)
axes[1].fill_between(range(len(orden_meses)), unidades_mes.values, alpha=0.2, color="#3498db")
axes[1].set_title("Unidades Vendidas por Mes", fontweight="bold")
axes[1].set_xlabel("Mes")
axes[1].set_ylabel("Unidades totales")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("analisis_ventas.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfica guardada como 'analisis_ventas.png'")